# Ablation 2026-09-01 — LDM + No Encoder + Temp only
**Run:** `abl_ldm_noenc_temp`  |  **W&B:** `1_Sep_2026_ldm_noenc_temp`  
**Model:** LDMModel (DDPM, unconditional)  |  **Fields:** `temperature` only


In [ ]:
RUN_NAME='abl_ldm_noenc_temp'; WANDB_RUN_NAME='1_Sep_2026_ldm_noenc_temp'
LDM_RUN_DIR='/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/abl_ldm_noenc_temp'
ENC_RUN_DIR=None
VAE_RUN_DIR='/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/vae_temp'
DATA_ROOT='/trace/group/forgelab/ngng/multifield/data_fields'
EVAL_OUT_DIR='/trace/group/forgelab/ngng/multifield/eval_results/ablation_20260901/abl_ldm_noenc_temp'
FIELD_NAMES=['temperature']; N_STEPS=3; DOWNSCALE_METHOD='direct'; NORMALIZE='standardize'
TIMESTEPS=1000; SCHEDULE='linear'; ENCODING=False; CONDITIONING='none'; DEVICE='cuda'
BATCH_INDEX=0; SAMPLE_INDEX=0; BATCH_SIZE=4; T_LIQ=1700.0; LIQ_THR=0.5
MELT_THRESHOLD=1900.0; ANALYSIS_CH=0; ANALYSIS_MAX_BATCH=None

See notebook 18 (`18_abl_ldm_noenc_sdf_results.ipynb`) for the full code — copy cells after this USER CONFIG. Only the paths and field config differ.

In [ ]:
%matplotlib inline
import os,sys,time; from pathlib import Path
import numpy as np,torch,pandas as pd; import matplotlib.pyplot as plt
from torch.utils.data import DataLoader; from scipy.ndimage import gaussian_filter as _gf
from IPython.display import display as _ipy_display
def _show(*a, **kw):
    for n in plt.get_fignums(): _ipy_display(plt.figure(n))
    plt.close('all')
plt.show = _show
if not torch.cuda.is_available() and DEVICE=='cuda': DEVICE='cpu'
def find_root(s=Path.cwd()):
    for p in [s,*s.parents]:
        if (p/'setup.py').exists() and (p/'diffusionsr').exists(): return p
    raise RuntimeError('no root')
PROJECT_ROOT=find_root()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))

In [ ]:
from diffusionsr.datasets.dataset import SimulationXZDataset
from diffusionsr.analysis.analysis_functions import get_profile
from diffusionsr.runners.train_ldm import LDMModel
def as_numpy(x): return x.detach().cpu().numpy() if isinstance(x,torch.Tensor) else np.asarray(x)
def mae_rmse(p,g): p,g=np.asarray(p).ravel(),np.asarray(g).ravel(); return {'MAE':float(np.mean(np.abs(p-g))),'RMSE':float(np.sqrt(np.mean((p-g)**2)))}
fn=FIELD_NAMES; has_sdf='sdfliqlabel' in fn; has_T='temperature' in fn; has_liq='liqlabel' in fn or has_sdf
kw=dict(downscale_method=DOWNSCALE_METHOD,root_folder=DATA_ROOT,normalize=NORMALIZE,n_steps=N_STEPS,field_names=FIELD_NAMES)
train_ds,dev_ds,test_ds=(SimulationXZDataset(split=s,**kw) for s in ['train','dev','test'])
model=LDMModel(vae_folder=VAE_RUN_DIR,results_folder=LDM_RUN_DIR,lr_encoder_folder=None,
    train_dataset=train_ds,dev_dataset=dev_ds,test_dataset=test_ds,
    timesteps=TIMESTEPS,conditioning=CONDITIONING,encoding=ENCODING,schedule=SCHEDULE,device=DEVICE,enc_output=False)
model.load_saved_model(); print(f'LDMModel (no encoder) loaded: {LDM_RUN_DIR}')

In [ ]:
STATS_CH = (N_STEPS - 1) * len(FIELD_NAMES)  # current-timestep temperature channel

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
maes, rmses, mp_errs, kh_errs, vc_maes, vae_maes = [], [], [], [], [], []
t0 = time.perf_counter(); n_samples = 0
for i, batch in enumerate(test_loader):
    if ANALYSIS_MAX_BATCH is not None and i >= ANALYSIS_MAX_BATCH: break
    _, hr_b, lr_b, ul_b = batch[:4]
    with torch.no_grad():
        samps = model.batch_sample(dataset=test_ds, batch=hr_b.to(DEVICE), x_e=None, sampler='DDPM')
        mu, _ = model.vae.encode(hr_b.to(DEVICE).float()); vr = model.vae.decode(mu).cpu().numpy()
    n_samples += hr_b.shape[0]
    for s in range(hr_b.shape[0]):
        p  = test_ds.unscale_data(samps[-1].cpu().numpy()[s], input_type='hr')
        g  = test_ds.unscale_data(as_numpy(hr_b[s]),          input_type='hr')
        vp = test_ds.unscale_data(vr[s],                      input_type='hr')
        p_curr = p[STATS_CH:STATS_CH+len(FIELD_NAMES)]
        g_curr = g[STATS_CH:STATS_CH+len(FIELD_NAMES)]
        m = mae_rmse(p_curr[0], g_curr[0]); maes.append(m['MAE']); rmses.append(m['RMSE'])
        vae_maes.append(mae_rmse(vp[STATS_CH], g_curr[0])['MAE'])
        try:
            pmp, pkh = get_profile(p_curr[0:1]); gmp, gkh = get_profile(g_curr[0:1])
            mp_errs.append(float(np.mean(np.abs(pmp-gmp)))); kh_errs.append(float(np.mean(np.abs(pkh-gkh))))
        except: pass
        vc_maes.append(float(np.mean(np.abs((p_curr[0]>MELT_THRESHOLD).astype(float)-(g_curr[0]>MELT_THRESHOLD).astype(float)))))
elapsed = time.perf_counter() - t0
print(f'n={len(maes)} DDPM (no enc)  MAE={np.nanmean(maes):.4f}±{np.nanstd(maes):.4f}')
print(f'  RMSE={np.nanmean(rmses):.4f}±{np.nanstd(rmses):.4f}')
print(f'  VAE recon MAE={np.nanmean(vae_maes):.4f}')
if mp_errs: print(f'  MP-MAE={np.nanmean(mp_errs):.2f}±{np.nanstd(mp_errs):.2f}px  KH-MAE={np.nanmean(kh_errs):.2f}px')
if vc_maes: print(f'  VC-MAE={np.nanmean(vc_maes):.4f}±{np.nanstd(vc_maes):.4f}')
import os; os.makedirs(EVAL_OUT_DIR, exist_ok=True)
summary = {'run_name':RUN_NAME,'wandb_run':WANDB_RUN_NAME,'model':'LDM','encoding':ENCODING,
           'conditioning':CONDITIONING,'fields':str(FIELD_NAMES),'sampler':'DDPM','n_test':len(maes),
           'mae_mean':float(np.nanmean(maes)),'mae_std':float(np.nanstd(maes)),
           'rmse_mean':float(np.nanmean(rmses)),'rmse_std':float(np.nanstd(rmses)),
           'vae_mae_mean':float(np.nanmean(vae_maes)) if vae_maes else float('nan'),
           'mp_mae_mean':float(np.nanmean(mp_errs)) if mp_errs else float('nan'),
           'kh_mae_mean':float(np.nanmean(kh_errs)) if kh_errs else float('nan')}
pd.DataFrame([summary]).to_csv(f'{EVAL_OUT_DIR}/metrics_summary.csv',index=False)
print(f'Saved -> {EVAL_OUT_DIR}/metrics_summary.csv')